In [10]:
import qutip as qt
from qutip import tensor, basis, qeye, Qobj
import numpy as np
from quantum_logical.trotter_diff import Trotterization
from tqdm import tqdm
from quantum_logical.state import state as st
from quantum_logical.cnot_gate_creation import cnot

# there is circular importation that needs fixed 

In [11]:
def erasure(values, rho, total_time, state_choice=None):
    T, dim, N, x_gate = values
    initial_state = qt.ptrace(rho, [0,1,2])
    ancilla_states = tensor(basis(dim, 0), basis(dim, 0), basis(dim, 0)) * tensor(basis(dim, 0), basis(dim, 0), basis(dim, 0)).dag()
    full_initial_state = tensor(initial_state, ancilla_states)

    # detection gate creation
    cnot1 = cnot(target=3, control=0, high=1, low=0, N=6)
    cnot2 = cnot(target=4, control=1, high=1, low=0, N=6)
    cnot3 = cnot(target=5, control=2, high=1, low=0, N=6)


    hada = qt.Qobj([[1/np.sqrt(2), 0, 1/np.sqrt(2)], [0, 1, 0], [1/np.sqrt(2), 0, -1/np.sqrt(2)]])
    # hada_layer = tensor(tensor([hada] * 3), tensor([qeye(dim)] * 3))

    gates = [cnot1, cnot2, cnot3]
    cnot_time = .5
    gate_times = [cnot_time, cnot_time, cnot_time]

    # running the circuit
    rho = full_initial_state

    for i in range(len(gates)):
        # if i == 10: # ensures that the hadamard can be fractionally broken and used 
        #     trotter_dt = gate_times[i] / 20
        #     trotter = Trotterization(trotter_dt=trotter_dt, T1=T[0], T2=T[1], dim=dim, num_qubits=3, qudit="qutrit")
        #     rho_enc = trotter.apply(rho=qt.ptrace(rho, [0,1,2]), duration=gate_times[i], unitary=[gates[i]], errors=False)
        #     rho = tensor(rho_enc[-1], 
        #                  tensor(basis(dim, 0), basis(dim, 0), basis(dim, 0)) 
        #                  * tensor(basis(dim, 0), basis(dim, 0), basis(dim, 0)).dag()) # Ensures dimensionality for future gates
        # else: 
        trotter_dt = gate_times[i] / 20
        trotter = Trotterization(trotter_dt=trotter_dt, T1=T[0], T2=T[1], dim=dim, num_qubits=N, qudit="qutrit")
        rho_enc = trotter.apply(rho=rho, duration=gate_times[i], unitary=[gates[i]], errors=False)
        rho = rho_enc[-1]
        total_time += gate_times[i]

    
    # measurement
    # measurement operators 
    proj = [tensor(qeye(dim), qeye(dim), qeye(dim), tensor(basis(dim, i), basis(dim, j), basis(dim, k))  
                   * tensor(basis(dim, i), basis(dim, j), basis(dim, k)).dag()) 
             for i in [0,1] for j in [0,1] for k in [0,1]]
    
    measurement_duration = 2
    trotter_dt = measurement_duration / 20
    trotter = Trotterization(trotter_dt=trotter_dt, T1=T[0], T2=T[1], dim=dim, num_qubits=N, qudit="qutrit")

    measurement_evo = trotter.apply(rho=rho, duration=measurement_duration, unitary=[tensor([qeye(dim)] * N)], errors=False)
    total_time += measurement_duration

    state_check = measurement_evo[-1]


    # projection results
    proj_res = [(proj * measurement_evo[-1]).tr() for proj in proj]
    proj_states = [(proj * measurement_evo[-1] * proj.dag()) for proj in proj]
   
    hada_layer_mod = tensor([hada] * 3)

    # Correction based on the results 
    # correction gates
    cnot1 =  cnot(target=2, control=0, high=2, low=0, N=3)
    cnot2 =  cnot(target=1, control=0, high=2, low=0, N=3)
    cnot3 =  cnot(target=0, control=1, high=2, low=0, N=3) 
    cnot4 =  cnot(target=2, control=1, high=2, low=0, N=3)
    cnot5 =  cnot(target=0, control=2, high=2, low=0, N=3)
    cnot6 =  cnot(target=1, control=2, high=2, low=0, N=3)
    
    # # correction_operators
    # r000 = r111 = [[hada_layer_mod * qt.tensor([qt.qeye(dim)] * 3) * hada_layer_mod.dag()]]
    # r001 = [[hada_layer_mod], [tensor(tensor([qeye(dim)] * 2), x_gate)], [cnot1], [hada_layer_mod]]
    # r010 = [[hada_layer_mod], [tensor(qeye(dim), x_gate, qeye(dim))], [cnot2], [hada_layer_mod]]
    # r011 = [[hada_layer_mod], [tensor(qeye(dim), x_gate, qeye(dim))], [tensor(tensor([qeye(dim)] * 2), x_gate)], [cnot2], [cnot1], [hada_layer_mod]]
    # r100 = [[hada_layer_mod], [tensor(x_gate, tensor([qeye(dim)] * 2))], [cnot3], [hada_layer_mod]]
    # r101 = [[hada_layer_mod], [tensor(x_gate, tensor([qeye(dim)] * 2))], [tensor(tensor([qeye(dim)] * 2), x_gate)], [cnot3], [cnot4], [hada_layer_mod]]
    # r110 = [[hada_layer_mod], [tensor(x_gate, tensor([qeye(dim)] * 2))], [tensor(qeye(dim), x_gate, qeye(dim))], [cnot5], [cnot6], [hada_layer_mod]]

    # recovery_ops = [r000, r001, r010, r011, r100, r101, r110, r111]
    # recovery_times = [[.5], [.03 ,.03, .5, .03], [.03, .03, .5, .03], [.03, .03, .03, .5, .5, .03], 
    #                   [.03, .03, .5, .03], [.03, .03, .03, .5, .5, .03], [.03, .03, .03, .5, .5, .03], [.5]]

    # correction_operators
    r000 = r111 = [[qt.tensor([qt.qeye(dim)] * 3)]]
    r001 = [[hada_layer_mod], [tensor(tensor([qeye(dim)] * 2), x_gate)], [cnot1], [qt.tensor([qt.qeye(dim)] * 3)], [hada_layer_mod]]
    r010 = [[hada_layer_mod], [tensor(qeye(dim), x_gate, qeye(dim))], [cnot2], [qt.tensor([qt.qeye(dim)] * 3)], [hada_layer_mod]]
    r011 = [[hada_layer_mod], [tensor(qeye(dim), x_gate, qeye(dim))], [tensor(tensor([qeye(dim)] * 2), x_gate)], [cnot2], [cnot1], [hada_layer_mod]]
    r100 = [[hada_layer_mod], [tensor(x_gate, tensor([qeye(dim)] * 2))], [cnot3], [qt.tensor([qt.qeye(dim)] * 3)], [hada_layer_mod]]
    r101 = [[hada_layer_mod], [tensor(x_gate, tensor([qeye(dim)] * 2))], [tensor(tensor([qeye(dim)] * 2), x_gate)], [cnot3], [cnot4], [hada_layer_mod]]
    r110 = [[hada_layer_mod], [tensor(x_gate, tensor([qeye(dim)] * 2))], [tensor(qeye(dim), x_gate, qeye(dim))], [cnot5], [cnot6], [hada_layer_mod]]

    recovery_ops = [r000, r001, r010, r011, r100, r101, r110, r111]
    recovery_times = [[1.12], [.03 , .03, .5, .53, .03], [.03, .03, .5, .53, .03], [.03, .03, .03, .5, .5, .03], 
                      [.03, .03, .5, .53, .03], [.03, .03, .03, .5, .5, .03], [.03, .03, .03, .5, .5, .03], [1.12]]

    # trotter_dt = gate_times[i] / 20
    # trotter = Trotterization(trotter_dt=trotter_dt, T1=T[0], T2=T[1], dim=dim, num_qubits=3, qudit="qutrit")
    # rho_enc = trotter.apply(rho=qt.ptrace(rho, [0,1,2]), duration=gate_times[i], unitary=[gates[i]], errors=False)
    # rho = tensor(rho_enc[-1], 
    #                 tensor(basis(dim, 0), basis(dim, 0), basis(dim, 0)) 
    #                 * tensor(basis(dim, 0), basis(dim, 0), basis(dim, 0)).dag()) # Ensures dimensionality for future gates


    # start the correction procedure 
    # correction_cycle 
    correction_duration = .5
    corrected_states = []
    for i in range(len(recovery_ops)):
        state_current = qt.ptrace(proj_states[i], [0,1,2])
        if proj_res[i] != 0:
            for j in range(len(recovery_ops[i])):
                trotter_dt = recovery_times[i][j] / 20
                trotter = Trotterization(trotter_dt=trotter_dt, T1=T[0], T2=T[1], dim=dim, num_qubits=3, qudit="qutrit")
                corrected_state = trotter.apply(state_current, duration=recovery_times[i][j], 
                                                unitary=recovery_ops[i][j], errors=False)
                state_current = corrected_state[-1]
            corrected_states.append(corrected_state)
        else:
            for j in range(len(recovery_ops[i])):
                if type(recovery_ops[i][j]) is not Qobj:
                    for k in range(len(recovery_ops[i][j])):
                        state_current = recovery_ops[i][j][k] * state_current * (recovery_ops[i][j][k]).dag()
                else:
                    state_current = recovery_ops[i][j] * state_current * (recovery_ops[i][j]).dag()
            corrected_states.append([state_current])
    total_time += correction_duration

    # combining the corrected states
    erasure_corrected_state = (sum([proj_res[j] * corrected_states[j][-1] for j in range(len(proj_res))]) / 
                               sum([proj_res[j] * corrected_states[j][-1] for j in range(len(proj_res))]).tr())

    
    return erasure_corrected_state, total_time, state_check

In [12]:
# declaring simulation variables
N = 6
dim = 3

In [13]:
from quantum_logical.gate_extender import Gate_extender

In [14]:
x_gate = qt.Qobj([[0, 1],[1, 0]])
gate_extention = Gate_extender(num_qubits=1)
x_gate = gate_extention.qubit_to_qudit(gate=x_gate, from_dim=2, to_dim=dim)

In [15]:
rho, state_vec = st(qubit_choices=["1","-","-"], dim=3, alpha=1, beta=0)

In [16]:
iterations = 1
# building T1 and T2 lists 
t1_list = np.linspace(100, 160, iterations)
t_list = []
for i in range(len(t1_list)):
    t2s = np.linspace(t1_list[i] * (2/3), t1_list[i] * (2/3), 1)
    for j in range(len(t2s)):
        t_list.append([t1_list[i], t2s[j]])

values = []
for i in range(iterations):
    values.append([t_list[i], dim, N, x_gate])


In [17]:
hada = qt.Qobj([[1/np.sqrt(2), 0, 1/np.sqrt(2)], [0, 1, 0], [1/np.sqrt(2), 0, -1/np.sqrt(2)]])
# vector setup 
basis0 = qt.Qobj([[1],[0],[0]])
basis1 = qt.Qobj([[0],[1],[0]])
basis2 = qt.Qobj([[0],[0],[1]])
vector0 = hada * basis0
vector1 = hada * basis1
vector2 = hada * basis2
vectors = [vector0, vector1, vector2]
vectors = [tensor(i,j,k) for i in vectors for j in vectors for k in vectors]

In [18]:
rho_encoded, state_vector_ = st(qubit_choices=["1", "-", "1"], dim=3, alpha=1, beta=0)
rho, total_time, state_check = erasure(values=values[0], rho=rho_encoded, total_time=0)
vals = []
for vec in vectors:
    val = (vec.dag() * qt.ptrace(rho, [0,1,2]) * vec)[0][0][0]
    vals.append(val)
vals

[0j,
 0j,
 0j,
 0j,
 0j,
 0j,
 0j,
 0j,
 0j,
 0j,
 0j,
 0j,
 0j,
 0j,
 0j,
 0j,
 0j,
 0j,
 0j,
 0j,
 0j,
 0j,
 0j,
 0j,
 0j,
 0j,
 (0.9999999999999999+0j)]

In [8]:
physical_error = []
logical_error = []
phase_errors = []
erasure_errors = []
t1 = []
t2 = []
physical_err = []

cycles = 50

state_choices = [[["-", "-", "-"], 1, 0], [["+", "+", "+"], 1, 0], 
                     [["+", "+", "+"], 1, 1], [["+", "+", "+"], 1, -1], 
                     [["+", "+", "+"], 1, 1j], [["+", "+", "+"], 1, -1j]]
# state_choices = [[["+", "+", "+"], 1, 0]]

proj_res = []

for state_choice in state_choices:
    log_err = []
    phys_err = []
    time = []
    proj_res_ = []
    eras_err = []
    phase_err = []
    physical_rate = []


    # this has to be the ugliest code you have ever written (fix this)
    if state_choice == [["-", "-", "-"], 1, 0]:
        vectors_phase = []
        vectors_erasure = []
        rho, state = st(qubit_choices=["-", "-", "+"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_phase.append(state)
        rho, state = st(qubit_choices=["-", "+", "-"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_phase.append(state)
        rho, state = st(qubit_choices=["+", "-", "-"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_phase.append(state)
        rho, state = st(qubit_choices=["-", "-", "1"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_erasure.append(state)
        rho, state = st(qubit_choices=["-", "1", "1"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_erasure.append(state)
        rho, state = st(qubit_choices=["1", "-", "-"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_erasure.append(state)
        rho, state = st(qubit_choices=["1", "-", "1"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_erasure.append(state)
        rho, state = st(qubit_choices=["1", "1", "-"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_erasure.append(state)
        rho, state = st(qubit_choices=["-", "1", "-"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_erasure.append(state)
    elif state_choice[0] == ["+", "+", "+"]:
        vectors_phase = []
        vectors_erasure = []
        rho, state = st(qubit_choices=["-", "+", "+"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_phase.append(state)
        rho, state = st(qubit_choices=["+", "+", "-"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_phase.append(state)
        rho, state = st(qubit_choices=["+", "-", "+"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_phase.append(state)
        rho, state = st(qubit_choices=["+", "+", "1"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_erasure.append(state)
        rho, state = st(qubit_choices=["+", "1", "1"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_erasure.append(state)
        rho, state = st(qubit_choices=["1", "+", "+"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_erasure.append(state)
        rho, state = st(qubit_choices=["1", "+", "1"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_erasure.append(state)
        rho, state = st(qubit_choices=["1", "1", "+"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_erasure.append(state)
        rho, state = st(qubit_choices=["+", "1", "+"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_erasure.append(state)

    for value in values:
    
        rho_encoded, state_vector_ = st(qubit_choices=state_choice[0], dim=3, alpha=state_choice[1], beta=state_choice[2])

        total_time = 0
        for i in tqdm(range(cycles)):
            rho_encoded, total_time = erasure(values=value, rho=rho_encoded, total_time=total_time)

            def neilson_fid(rho, sigma):
                return (((rho.sqrtm()) * sigma * (rho.sqrtm())).sqrtm()).tr()

            logical = [state_vector_]
            vals = []
            for vec in logical:
                val = (vec.dag() * qt.ptrace(rho_encoded, [0,1,2]) * vec)[0][0][0]
                vals.append(val)
            log_err.append(np.abs(1 - np.abs(sum(vals))))

            vals = []
            for vec in vectors_phase:
                val = (vec.dag() * qt.ptrace(rho_encoded, [0,1,2]) * vec)[0][0][0]
                vals.append(val)
            phase_err.append(np.abs(sum(vals)))

            vals = []
            for vec in vectors_erasure:
                val = (vec.dag() * qt.ptrace(rho_encoded, [0,1,2]) * vec)[0][0][0]
                vals.append(val)
            eras_err.append(np.abs(sum(vals)))

            vals = []
            for vec in vectors:
                val = (vec.dag() * qt.ptrace(rho_encoded, [0,1,2]) * vec)[0][0][0]
                vals.append(val)

            t_phase = (2 * value[0][0] * value[0][1])/(2 * value[0][0] - value[0][1])
            physical_error_val = ((1 - np.exp((-total_time) * ((1/value[0][0])))) + (1 - np.exp((-total_time) * ((1/t_phase)))) 
                                - (1 - np.exp((-total_time) * ((1/value[0][0])))) * (1 - np.exp((-total_time) * ((1/t_phase)))))
            phys_err.append(np.abs(physical_error_val))
            time.append(total_time)
            phys_rate = 1/(value[0][0]) + 1/((2*value[0][0]*value[0][1])/(2*value[0][0] - value[0][1]))
            physical_rate.append(phys_rate)
            # print(state_vector_.dag() * qt.ptrace(rho_encoded, [0,1,2]) * state_vector_)

        phase_errors.append(phase_err)
        erasure_errors.append(eras_err)
        physical_error.append(phys_err)
        logical_error.append(log_err)

  0%|          | 0/50 [00:00<?, ?it/s]

100%|██████████| 50/50 [04:42<00:00,  5.65s/it]


In [9]:
logical_error

[[0.014836618555477799,
  0.020397533009460234,
  0.02587517190826616,
  0.03131162319328595,
  0.036707578838022226,
  0.042063415180085606,
  0.04737950137514302,
  0.052656201992817175,
  0.057893877203682886,
  0.0630928828817745,
  0.06825357064205095,
  0.07337628784353156,
  0.07846137777161066,
  0.0835091796248012,
  0.08852002857533603,
  0.09349425591481353,
  0.09843218895474037,
  0.10333415124747736,
  0.10820046252557769,
  0.11303143881936273,
  0.1178273925443113,
  0.12258863244422746,
  0.12731546375686797,
  0.1320081882067996,
  0.13666710408486382,
  0.141292506274597,
  0.1458846863442631,
  0.1504439324953173,
  0.15497052976773718,
  0.1594647599816612,
  0.16392690180371816,
  0.16835723078945086,
  0.1727560194241886,
  0.17712353720916207,
  0.18146005064034987,
  0.18576582332286473,
  0.19004111592145678,
  0.19428618631064487,
  0.1985012895388537,
  0.2026866778918266,
  0.20684260093658646,
  0.21096930555093496,
  0.21506703597737575,
  0.2191360338237

In [10]:
# sort data
phys = []
logic = []
e_errors = []
p_errors = []

# for i in range(len(values)):
#     phys.append(sum([physical_error[j][i] for j in range(len(state_choices))]) / len(state_choices))
#     logic.append(sum([logical_error[j][i] for j in range(len(state_choices))]) / len(state_choices))
#     e_errors.append(sum([erasure_errors[j][i] for j in range(len(state_choices))]) / len(state_choices))
#     p_errors.append(sum([phase_errors[j][i] for j in range(len(state_choices))]) / len(state_choices))
for i in range(cycles):
    phys.append(sum([physical_error[j][i] for j in range(len(state_choices))]) / len(state_choices))
    logic.append(sum([logical_error[j][i] for j in range(len(state_choices))]) / len(state_choices))
    e_errors.append(sum([erasure_errors[j][i] for j in range(len(state_choices))]) / len(state_choices))
    p_errors.append(sum([phase_errors[j][i] for j in range(len(state_choices))]) / len(state_choices))

In [11]:
logic

[0.016088700209922202,
 0.028841773842248303,
 0.04130230021386797,
 0.05347494081951565,
 0.06536781067969544,
 0.07698878919422336,
 0.08834551523160311,
 0.09944539580916618,
 0.11029561381652532,
 0.12090313481529082,
 0.13127471364718546,
 0.1414169012385628,
 0.15133605084937318,
 0.16103832421563172,
 0.17052969749180277,
 0.17981596708446398,
 0.18890275513833577,
 0.19779551501695872,
 0.20649953650653521,
 0.21501995092200787,
 0.2233617360147766,
 0.2315297207066933,
 0.23952858973541113,
 0.2473628880983942,
 0.25503702539870227,
 0.26255528001579087,
 0.2699218031857575,
 0.2771406228872637,
 0.28421564771980845,
 0.2911506705598021,
 0.2979493721064757,
 0.3046153243785099,
 0.3111519940340744,
 0.3175627456653875,
 0.3238508448957043,
 0.33001946145025113,
 0.3360716720930224,
 0.3420104635113584,
 0.34783873507024815,
 0.3535593014904002,
 0.359174895439377,
 0.36468817007196613,
 0.3701017014641807,
 0.375417990955605,
 0.38063946745509813,
 0.38576848964881844,
 0.390

In [12]:
e_errors

[0.0031012363790004333,
 0.0031429501870822896,
 0.0031858033307012694,
 0.0032264357969885096,
 0.0032649178597677717,
 0.003301326378650458,
 0.0033357341910705187,
 0.003368211428628032,
 0.003398826142400774,
 0.0034276441816897484,
 0.003454729081832753,
 0.00348014240167481,
 0.0035039436961089234,
 0.0035261905552643412,
 0.003546938673678134,
 0.003566241930695068,
 0.0035841524351301036,
 0.003600720576768786,
 0.0036159950867805297,
 0.0036300231019414666,
 0.0036428502084569366,
 0.003654520491188078,
 0.0036650765720563903,
 0.0036745596712804553,
 0.0036830096498136956,
 0.0036904650475392685,
 0.0036969631351649406,
 0.0037025399326483257,
 0.0037072302653456666,
 0.003711067827616323,
 0.003714085174589781,
 0.0037163137792804953,
 0.0037177840594047715,
 0.00371852543558436,
 0.0037185663380945415,
 0.003717934239704216,
 0.0037166556957627874,
 0.003714756383481591,
 0.0037122611119973416,
 0.0037091938508233414,
 0.003705577761350241,
 0.0037014352271611995,
 0.003696

In [13]:
p_errors

[0.005554106164127364,
 0.010989770717211746,
 0.016305525813316273,
 0.021504541179227562,
 0.0265900783471794,
 0.031565298678039755,
 0.036433265451845186,
 0.04119694703572211,
 0.045859219902516325,
 0.05042287156507263,
 0.054890603442562184,
 0.05926503360213118,
 0.06354869942516174,
 0.06774406019148094,
 0.07185349957982486,
 0.07587932807391756,
 0.07982378532527616,
 0.08368904238716245,
 0.08747720392329934,
 0.09119031032962156,
 0.0948303397770736,
 0.09839921020206166,
 0.10189878124148333,
 0.10533085607444337,
 0.10869718323521599,
 0.11199945835039636,
 0.11523932582449868,
 0.11841838048637637,
 0.1215381691506033,
 0.12460019215476777,
 0.12760590483919257,
 0.13055671897771395,
 0.13345400416932057,
 0.13629908916013878,
 0.13909326317389933,
 0.1418377771451961,
 0.14453384493649862,
 0.14718264451644025,
 0.14978531910684234,
 0.1523429782702453,
 0.15485669899579338,
 0.15732752671478734,
 0.15975647630459008,
 0.16214453305910617,
 0.16449265361923002,
 0.1668

In [14]:
import csv

# Writing the arrays to a CSV file
with open('3rd_order_states_erasure.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(phys)  
    writer.writerow(logic)
    writer.writerow(time)
    writer.writerow(e_errors)  
    writer.writerow(p_errors)   
    writer.writerow(physical_rate)   